# Kernel Mechanism Walkthrough

This notebook starts **exactly where** [inverse_problem_walkthrough.ipynb](./inverse_problem_walkthrough.ipynb) ends.

That notebook finishes by producing three toy slice graphs and their WL feature matrix:

- `shared_signal`
- `same_circuit`
- `other_circuit`

Here we reuse that same toy setup and answer the next question:

> Once we already have a WL matrix, what do the classical kernels actually do with it?

The notebook focuses only on the kernel stage:

1. rebuild the same toy WL matrix from the inverse-problem notebook,
2. verify the bridge with the old cosine-similarity view,
3. show how the **linear** and **RBF** kernels turn WL vectors into similarities,
4. and make the classifier geometry visible using small perturbations around those same WL rows.


## Bridge between the two notebooks

The important relationship is:

- `inverse_problem_walkthrough.ipynb` explains how we get from an observed patch-effect matrix to a graph and then to a WL matrix.
- this notebook assumes that WL matrix is already available and explains what the **kernel layer** is doing next.

So this notebook is not a different toy example. It is the **continuation** of the same toy example.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel, rbf_kernel
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from pig.embeddings import compute_wl_features
from pig.graph import GraphBuilder
from pig.kernels import ClassicalKernelClassifier
from pig.patching import ComponentSpec, PatchEffectDataset, PatchEffectTensor
from pig.prompts import PromptPair, SliceLabel

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 200,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.frameon": False,
    }
)

rng = np.random.default_rng(7)


## Stage 1. Rebuild the exact toy WL matrix from the inverse-problem notebook

These are the same toy effect matrices used at the end of `inverse_problem_walkthrough.ipynb`.

We are not re-explaining why they exist; we are only reconstructing the same endpoint so the two notebooks line up exactly.


### How to read the WL matrix

Before looking at the heatmap, here is what each part means:

- **Rows**: each row is one toy slice graph:
  - `shared_signal`
  - `same_circuit`
  - `other_circuit`
- **Columns**: each column is one **active WL feature**.
  In the notebook they appear as `f9`, `f21`, `f22`, etc.
  These are short column ids taken from the full WL vocabulary.
- **Cell value**: the number inside each cell is the **count** of that WL feature in that slice graph.
  In this toy example the counts are mostly `0` or `1`.
- **Color**: darker blue means a larger WL feature count.
- **Color bar**: maps the heatmap color back to the numeric WL count.

So the whole matrix should be read as:

> each row is one graph, each column is one structural WL pattern, and each cell says how strongly that pattern appears in that graph.

That is exactly the object the kernel stage consumes next.


In [ ]:
node_labels = ["L0T0", "L0T1", "L1T0", "L1T1"]
component_axis = [ComponentSpec(node_type="res")]

def make_dataset(matrix, slice_label):
    ds = PatchEffectDataset()
    for i in range(matrix.shape[0]):
        pair = PromptPair(
            x_cln=f"clean-{slice_label.corruption}-{i}",
            x_crp=f"corrupt-{slice_label.corruption}-{i}",
            y_star="target",
            slice_label=slice_label,
            meta={},
        )
        effects = matrix[i].reshape(2, 2, 1).astype(np.float32)
        ds.add(
            PatchEffectTensor(
                effects=effects,
                component_axis=component_axis,
                prompt_pair=pair,
                base_score=-1.0,
                clean_score=1.0,
            )
        )
    return ds

X_obs = np.array([
    [ 2.0,  1.8, -1.6,  0.1],
    [ 1.0,  0.9, -0.8, -0.1],
    [ 0.0,  0.1,  0.0,  0.2],
    [-1.0, -0.8,  0.7,  0.0],
    [-2.0, -1.7,  1.5, -0.2],
    [-1.0, -0.9,  0.8,  0.1],
], dtype=np.float32)

X_same_circuit = np.array([
    [ 1.8,  1.6, -1.5,  0.1],
    [ 0.9,  0.8, -0.8,  0.0],
    [ 0.1,  0.0, -0.1,  0.1],
    [-0.9, -0.8,  0.7,  0.0],
    [-1.8, -1.6,  1.4, -0.1],
    [-0.9, -0.8,  0.7,  0.1],
], dtype=np.float32)

X_other_circuit = np.array([
    [ 1.8,  0.1, -0.2,  1.6],
    [ 0.9,  0.0, -0.1,  0.8],
    [ 0.0,  0.1,  0.0,  0.1],
    [-0.9,  0.1,  0.2, -0.8],
    [-1.8, -0.1,  0.2, -1.6],
    [-0.9,  0.0,  0.1, -0.8],
], dtype=np.float32)

slice_obs = SliceLabel(task="inverse_demo", corruption="shared_signal")
slice_same = SliceLabel(task="inverse_demo", corruption="same_circuit")
slice_other = SliceLabel(task="inverse_demo", corruption="other_circuit")

builder = GraphBuilder(k=2, enforce_direction=True)
graph_obs = builder.build_from_slice(make_dataset(X_obs, slice_obs), slice_obs)
graph_same = builder.build_from_slice(make_dataset(X_same_circuit, slice_same), slice_same)
graph_other = builder.build_from_slice(make_dataset(X_other_circuit, slice_other), slice_other)

graphs = {
    slice_obs: graph_obs,
    slice_same: graph_same,
    slice_other: graph_other,
}
feature_matrix = compute_wl_features(graphs, depth=2)
X_wl = feature_matrix.to_matrix()
slice_names = [s.corruption for s in feature_matrix.slice_labels]

feature_var = X_wl.var(axis=0)
active_idx = [idx for idx in np.argsort(feature_var)[::-1] if feature_var[idx] > 0][:12]
if not active_idx:
    active_idx = list(range(min(12, X_wl.shape[1])))
X_wl_view = X_wl[:, active_idx]
feature_labels = [f"f{idx}" for idx in active_idx]

print("WL matrix shape:", X_wl.shape)
print("Slice order:", slice_names)

fig, ax = plt.subplots(figsize=(9.5, 3.8))
im = ax.imshow(X_wl_view, aspect="auto", cmap="Blues")
ax.set_xticks(np.arange(len(feature_labels)))
ax.set_xticklabels(feature_labels, rotation=35, ha="right")
ax.set_yticks(np.arange(len(slice_names)))
ax.set_yticklabels(slice_names)
ax.set_title("Same toy WL matrix we reached in the inverse-problem notebook")
for i in range(X_wl_view.shape[0]):
    for j in range(X_wl_view.shape[1]):
        ax.text(j, i, f"{int(X_wl_view[i, j])}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="WL feature count")
plt.tight_layout()
plt.show()


## Stage 2. Verify the bridge with the old similarity view

In the previous notebook, the final view was a **cosine similarity** over WL features.

We start by reproducing that same lens, so the handoff between notebooks is explicit.


In [ ]:
K_cosine = cosine_similarity(X_wl)

fig, ax = plt.subplots(figsize=(5.8, 4.8))
im = ax.imshow(K_cosine, cmap="magma", vmin=0.0, vmax=1.0)
ax.set_xticks(np.arange(len(slice_names)))
ax.set_xticklabels(slice_names, rotation=35, ha="right")
ax.set_yticks(np.arange(len(slice_names)))
ax.set_yticklabels(slice_names)
ax.set_title("Cosine similarity over the same toy WL matrix")
for i in range(K_cosine.shape[0]):
    for j in range(K_cosine.shape[1]):
        ax.text(j, i, f"{K_cosine[i, j]:.2f}", ha="center", va="center", fontsize=10, color="white" if K_cosine[i, j] < 0.55 else "black")
plt.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()

print("Pairwise cosine similarities")
for i, a in enumerate(slice_names):
    for j, b in enumerate(slice_names):
        print(f"  {a:>13} vs {b:<13}: {K_cosine[i, j]:.3f}")


## Stage 3. What changes when we move from cosine to the actual classical kernels

The repo's classical kernel path does not use cosine similarity as the classifier itself.

It uses:

- `StandardScaler`
- `SVC(kernel="linear")` or `SVC(kernel="rbf")`

So the next step is to take this same WL matrix and look at the similarities induced by those kernels.


In [ ]:
scaler = StandardScaler()
X_wl_scaled = scaler.fit_transform(X_wl)

scaled_view = X_wl_scaled[:, active_idx]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
im0 = axes[0].imshow(X_wl_view, aspect="auto", cmap="Blues")
axes[0].set_title("Raw active WL features")
axes[0].set_xticks(np.arange(len(feature_labels)))
axes[0].set_xticklabels(feature_labels, rotation=35, ha="right")
axes[0].set_yticks(np.arange(len(slice_names)))
axes[0].set_yticklabels(slice_names)
for i in range(X_wl_view.shape[0]):
    for j in range(X_wl_view.shape[1]):
        axes[0].text(j, i, f"{int(X_wl_view[i, j])}", ha="center", va="center", fontsize=9)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

vmax = float(np.max(np.abs(scaled_view)))
im1 = axes[1].imshow(scaled_view, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("Same features after StandardScaler")
axes[1].set_xticks(np.arange(len(feature_labels)))
axes[1].set_xticklabels(feature_labels, rotation=35, ha="right")
axes[1].set_yticks(np.arange(len(slice_names)))
axes[1].set_yticklabels(slice_names)
for i in range(scaled_view.shape[0]):
    for j in range(scaled_view.shape[1]):
        axes[1].text(j, i, f"{scaled_view[i, j]:+.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## Stage 4. Linear kernel on the inverse-problem WL matrix

The linear kernel is just a dot product between scaled WL rows:

$$
K_{\text{linear}}(x_i, x_j) = x_i^\top x_j
$$

So it measures whether two slices point in a similar direction in WL feature space.


In [ ]:
K_linear = linear_kernel(X_wl_scaled, X_wl_scaled)

fig, ax = plt.subplots(figsize=(5.8, 4.8))
im = ax.imshow(K_linear, cmap="RdBu_r")
ax.set_xticks(np.arange(len(slice_names)))
ax.set_xticklabels(slice_names, rotation=35, ha="right")
ax.set_yticks(np.arange(len(slice_names)))
ax.set_yticklabels(slice_names)
ax.set_title("Linear kernel similarity on the same WL matrix")
for i in range(K_linear.shape[0]):
    for j in range(K_linear.shape[1]):
        ax.text(j, i, f"{K_linear[i, j]:+.1f}", ha="center", va="center", fontsize=10, color="white" if abs(K_linear[i, j]) > 7 else "black")
plt.colorbar(im, ax=ax, label="linear-kernel similarity")
plt.tight_layout()
plt.show()

print("Linear-kernel similarities")
for i, a in enumerate(slice_names):
    for j, b in enumerate(slice_names):
        print(f"  {a:>13} vs {b:<13}: {K_linear[i, j]:+.3f}")


## Stage 5. RBF kernel on the same WL matrix

The RBF kernel uses distance:

$$
K_{\text{RBF}}(x_i, x_j) = \exp\left(-\gamma \lVert x_i - x_j \rVert^2\right)
$$

This makes the notion of similarity more local. Two slices only look similar if their WL rows are truly close.


In [ ]:
gamma = 0.02
K_rbf = rbf_kernel(X_wl_scaled, X_wl_scaled, gamma=gamma)

fig, ax = plt.subplots(figsize=(5.8, 4.8))
im = ax.imshow(K_rbf, cmap="magma", vmin=0.0, vmax=1.0)
ax.set_xticks(np.arange(len(slice_names)))
ax.set_xticklabels(slice_names, rotation=35, ha="right")
ax.set_yticks(np.arange(len(slice_names)))
ax.set_yticklabels(slice_names)
ax.set_title(f"RBF kernel similarity on the same WL matrix (gamma={gamma})")
for i in range(K_rbf.shape[0]):
    for j in range(K_rbf.shape[1]):
        ax.text(j, i, f"{K_rbf[i, j]:.2f}", ha="center", va="center", fontsize=10, color="white" if K_rbf[i, j] < 0.55 else "black")
plt.colorbar(im, ax=ax, label="RBF similarity")
plt.tight_layout()
plt.show()

print("RBF-kernel similarities")
for i, a in enumerate(slice_names):
    for j, b in enumerate(slice_names):
        print(f"  {a:>13} vs {b:<13}: {K_rbf[i, j]:.3f}")


## Stage 6. The same three slices, now seen through three similarity lenses

This is the cleanest way to relate the notebooks:

- the inverse-problem notebook ends with the **cosine** view,
- this notebook shows what the **linear** and **RBF** kernel views do to the same rows.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.2))
for ax, K, title, cmap, vmin, vmax in [
    (axes[0], K_cosine, "Cosine", "magma", 0.0, 1.0),
    (axes[1], K_linear, "Linear kernel", "RdBu_r", None, None),
    (axes[2], K_rbf, "RBF kernel", "magma", 0.0, 1.0),
]:
    im = ax.imshow(K, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(slice_names)))
    ax.set_xticklabels(slice_names, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(slice_names)))
    ax.set_yticklabels(slice_names)
    ax.set_title(title)
    for i in range(K.shape[0]):
        for j in range(K.shape[1]):
            label = f"{K[i, j]:+.1f}" if title == "Linear kernel" else f"{K[i, j]:.2f}"
            color = "white" if (title == "Linear kernel" and abs(K[i, j]) > 7) or (title != "Linear kernel" and K[i, j] < 0.55) else "black"
            ax.text(j, i, label, ha="center", va="center", fontsize=9, color=color)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("Three similarity views over the exact same toy WL matrix", y=1.02)
plt.tight_layout()
plt.show()


## Stage 7. Make the classifier geometry visible

With only three slices, the kernel matrices are easy to read, but they are not great for visualizing a decision boundary.

So here we do something purely pedagogical:

- keep the exact three WL rows from the inverse-problem notebook as **anchors**,
- generate small perturbation copies around them,
- and define two families:
  - **reuse-family**: perturbations around `shared_signal` and `same_circuit`
  - **other-family**: perturbations around `other_circuit`

That lets us see the geometry that the kernels are exploiting, while still staying anchored to the exact toy WL rows from the previous notebook.


In [ ]:
anchor_lookup = {name: X_wl[idx] for idx, name in enumerate(slice_names)}

augment_specs = [
    ("shared_signal", 0, 4, 0.22),
    ("same_circuit", 0, 4, 0.22),
    ("other_circuit", 1, 5, 0.22),
]

X_aug = []
y_aug = []
point_names = []
for anchor_name, label, copies, noise_scale in augment_specs:
    anchor = anchor_lookup[anchor_name]
    for copy_idx in range(copies):
        noise = rng.normal(scale=noise_scale, size=anchor.shape)
        X_aug.append(anchor + noise)
        y_aug.append(label)
        point_names.append(f"{anchor_name[:2]}-{copy_idx+1}")

X_aug = np.asarray(X_aug, dtype=np.float32)
y_aug = np.asarray(y_aug, dtype=np.int32)
class_names = {0: "reuse-family", 1: "other-family"}

print("Augmented WL matrix shape:", X_aug.shape)
print("Class balance:", {class_names[int(k)]: int((y_aug == k).sum()) for k in np.unique(y_aug)})


## Stage 8. Linear vs RBF decision geometry

We project the augmented WL cloud to 2D with PCA.
This is just for visualization; the real classifier still works in full WL space.


In [ ]:
scaler_aug = StandardScaler()
X_aug_scaled = scaler_aug.fit_transform(X_aug)
X_anchor_scaled = scaler_aug.transform(X_wl)

pca = PCA(n_components=2, random_state=0)
X_2d = pca.fit_transform(X_aug_scaled)
X_anchor_2d = pca.transform(X_anchor_scaled)

linear_vis = SVC(kernel="linear", C=1.0)
rbf_vis = SVC(kernel="rbf", C=1.0, gamma=0.8)
linear_vis.fit(X_2d, y_aug)
rbf_vis.fit(X_2d, y_aug)

def plot_decision_boundary(ax, model, title):
    x_min, x_max = X_2d[:, 0].min() - 1.0, X_2d[:, 0].max() + 1.0
    y_min, y_max = X_2d[:, 1].min() - 1.0, X_2d[:, 1].max() + 1.0
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 320),
        np.linspace(y_min, y_max, 320),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.decision_function(grid).reshape(xx.shape)

    ax.contourf(xx, yy, zz > 0, alpha=0.16, levels=1, colors=["#8ecae6", "#f4a261"])
    ax.contour(xx, yy, zz, levels=[0], colors="black", linewidths=1.5)

    colors = np.where(y_aug == 0, "#1d3557", "#d62828")
    for i in range(len(X_2d)):
        ax.scatter(X_2d[i, 0], X_2d[i, 1], s=70, color=colors[i], edgecolor="white", linewidth=0.9)

    anchor_colors = ["#0f766e", "#0f766e", "#b91c1c"]
    for i, name in enumerate(slice_names):
        ax.scatter(
            X_anchor_2d[i, 0],
            X_anchor_2d[i, 1],
            s=180,
            marker="X",
            color=anchor_colors[i],
            edgecolor="white",
            linewidth=1.2,
            zorder=5,
        )
        ax.text(X_anchor_2d[i, 0] + 0.08, X_anchor_2d[i, 1] + 0.08, name, fontsize=9, fontweight="bold")

    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
plot_decision_boundary(axes[0], linear_vis, "Linear SVM over the anchored WL cloud")
plot_decision_boundary(axes[1], rbf_vis, "RBF SVM over the anchored WL cloud")
plt.tight_layout()
plt.show()


## Stage 9. Use the same classifier object as the repo

Finally, we run the actual `ClassicalKernelClassifier` object on the augmented WL cloud.

That closes the loop:

- the **anchor rows** come from the inverse-problem notebook,
- the **geometry** comes from small perturbations around those rows,
- and the **classifier** is the same object used by the repo's classical baseline.


In [ ]:
linear_clf = ClassicalKernelClassifier(kernel="linear", random_state=42, normalize=True)
rbf_clf = ClassicalKernelClassifier(kernel="rbf", random_state=42, normalize=True)

linear_cv = linear_clf.cross_validate(X_aug, y_aug, cv=4)
rbf_cv = rbf_clf.cross_validate(X_aug, y_aug, cv=4)

print("Linear CV:", linear_cv)
print("RBF CV:", rbf_cv)


## What this notebook demonstrates

Relative to `inverse_problem_walkthrough.ipynb`, the message is now:

1. the previous notebook gave us a toy WL matrix over slices,
2. this notebook reuses that exact WL matrix,
3. cosine, linear-kernel, and RBF-kernel views are three different similarity lenses over the same rows,
4. and the classical kernel stage works by turning those structural similarities into a decision rule.

So the kernel stage is not a separate mystery.
It is simply the **next layer of comparison** built on top of the toy WL representation from the inverse-problem notebook.
